In [ ]:
import gzip
import ast
import pickle
from collections import defaultdict, Counter
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm


# Paths

INTERACTIONS_PATH = "(steam)australian_users_items.json.gz"
REVIEWS_PATH = "(steam)australian_user_reviews.json.gz"  
ITEM_TEXT_PATH = "steam_item_text.pkl"#output path
SBERT_EMB_PATH = "steam_item_sbert.pt"#outputh path

# Hyperparameters

MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5
EMB_DIM = 384
BATCH_SIZE = 512
BERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


#  Load user-item interactions

print("Loading Steam interactions")

user_items_raw = defaultdict(list)
item_names = {}

with gzip.open(INTERACTIONS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        user_obj = ast.literal_eval(line)
        uid = user_obj["user_id"]

        for item in user_obj.get("items", []):
            iid = item["item_id"]
            val = item.get("playtime_forever", 1.0)
            ts = item.get("timestamp", 0)

            user_items_raw[uid].append((iid, ts, val))

            if iid not in item_names:
                item_names[iid] = item.get("item_name", "unknown item")

print(f"Loaded {len(user_items_raw)} users, {len(item_names)} items")



def k_core_filter(user_items, min_user=MIN_USER_INTERACTIONS, min_item=MIN_ITEM_INTERACTIONS):
    print("Applying k-core filtering...")

    while True:
        user_counts = {u: len(v) for u, v in user_items.items()}
        item_counts = Counter(iid for seq in user_items.values() for iid, _, _ in seq)

        filtered_users = {
            u: seq for u, seq in user_items.items()
            if len(seq) >= min_user
        }

        filtered_users_items = {}
        for u, seq in filtered_users.items():
            new_seq = [
                (iid, ts, val)
                for iid, ts, val in seq
                if item_counts[iid] >= min_item
            ]

            if len(new_seq) >= min_user:
                filtered_users_items[u] = new_seq

        if len(filtered_users_items) == len(user_items):
            break

        user_items = filtered_users_items

    return user_items


user_items_filtered = k_core_filter(user_items_raw)
items_set = {iid for seq in user_items_filtered.values() for iid, _, _ in seq}

print(f"After k-core: {len(user_items_filtered)} users, {len(items_set)} items")


print("Sorting user sequences by chronologically")

user_items_sorted = {}

for u, seq in user_items_filtered.items():
    seq_sorted = sorted(seq, key=lambda x: x[1])
    user_items_sorted[u] = [(iid, val) for iid, _, val in seq_sorted]

print("User sequences sorted.")


#  Build item text

print("Building item text for SBERT")

items = sorted(items_set)
item_texts = {}

for iid in items:
    parts = [item_names.get(iid, "unknown item")]
    text = ". ".join(parts)
    item_texts[iid] = text

with open(ITEM_TEXT_PATH, "wb") as f:
    pickle.dump(item_texts, f)

print(f"Item text for {len(item_texts)} items saved.")

# 5️Encode SBERT embeddings

print("Encoding item texts with SBERT")

sbert = SentenceTransformer(BERT_MODEL).to(device)
item_text_list = [item_texts[iid] for iid in items]

embs_list = []

for i in tqdm(range(0, len(item_text_list), BATCH_SIZE)):
    batch = item_text_list[i:i+BATCH_SIZE]

    emb = sbert.encode(
        batch,
        convert_to_tensor=True,
        device=device,
        normalize_embeddings=True
    )

    embs_list.append(emb)

item_embs = torch.cat(embs_list, dim=0)

# Project if dimension mismatch
if item_embs.size(1) != EMB_DIM:
    projector = torch.nn.Linear(item_embs.size(1), EMB_DIM, bias=False).to(device)
    torch.nn.init.xavier_uniform_(projector.weight)
    item_embs = projector(item_embs)

item_embs = torch.nn.functional.normalize(item_embs)

torch.save(item_embs.cpu(), SBERT_EMB_PATH)

print(f"SBERT embeddings saved: {item_embs.shape}")

